# Debug Notebook: GPU Optimization Benchmark for Qwen3 4B / 0.6B

This notebook benchmarks speculative decoding for the Qwen3 review setup with GPU optimization switches OFF vs ON,
then produces an automatic delta table for:
- draft time (`draft_elapsed_s`)
- verify time (`verify_elapsed_s`)
- end-to-end latency (`latency_s`)

Outputs are saved under `Review/results/` for the Qwen3 4B target and Qwen3 0.6B draft configuration.

In [1]:
import hashlib
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'torch is required for this notebook. Select a Python environment with PyTorch installed.'
    ) from exc

# Resolve project root robustly (works from notebook root or subfolders).
search_starts = []
if '__file__' in globals():
    search_starts.append(Path(__file__).resolve().parent)
search_starts.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_starts:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'speculative.py').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/speculative.py')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior for reproducibility in debug sessions.
os.environ.setdefault('SPECDEC_HF_OFFLINE_FIRST', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for this benchmark notebook.')

import speculative as spec_mod
from config import DRAFT_MODELS, DRAFT_QUANT, TARGET_MODEL_ID, TARGET_QUANT, REGIMES
from speculative import load_model_on_device
from utils import set_seed

results_dir = project_root / 'Review' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
manifests_dir = project_root / 'manifests'

print(f'Project root: {project_root}')
print(f'CUDA device count: {torch.cuda.device_count()}')

Project root: C:\Working\speculative-decoding-main_v10\speculative-decoding-main
CUDA device count: 2


In [2]:
import importlib
import os
import sys

review_root = project_root / "Review"
review_results_dir = review_root / "results"
review_figures_dir = review_root / "figures"
review_stability_dir = review_results_dir / "stability"
review_drifter_dir = review_results_dir / "drifter_ckpt"

qwen3_target_model_id = "Qwen/Qwen3-4B"
qwen3_draft_model_id = "Qwen/Qwen3-0.6B"

os.environ["SPECDEC_TARGET_MODEL_ID"] = qwen3_target_model_id
os.environ["SPECDEC_DRAFT_MODEL_ID"] = qwen3_draft_model_id
os.environ["SPECDEC_RESULTS_DIR"] = str(review_results_dir)
os.environ["SPECDEC_STABILITY_DIR"] = str(review_stability_dir)
os.environ["SPECDEC_FIGURES_DIR"] = str(review_figures_dir)
os.environ["SPECDEC_DRIFTER_CHECKPOINT_DIR"] = str(review_drifter_dir)
os.environ["SPECDEC_TARGET_QUANT"] = "fp16"
os.environ["SPECDEC_DRAFT_QUANT"] = "fp16"
os.environ["SPECDEC_ACSD_PRIMARY_DRAFT"] = "0.6B"
os.environ["SPECDEC_ACSD_RESCUE_DRAFT"] = "0.6B"

config_module = sys.modules.get("config")
if config_module is None:
    import config as config_module
config_module = importlib.reload(config_module)

globals().update({
    "TARGET_MODEL_ID": config_module.TARGET_MODEL_ID,
    "DRAFT_MODELS": config_module.DRAFT_MODELS,
    "RESULTS_DIR": config_module.RESULTS_DIR,
    "STABILITY_DIR": config_module.STABILITY_DIR,
    "FIGURES_DIR": config_module.FIGURES_DIR,
    "TARGET_QUANT": config_module.TARGET_QUANT,
    "DRAFT_QUANT": config_module.DRAFT_QUANT,
})
if hasattr(config_module, "ACSD_EVAL"):
    globals()["ACSD_EVAL"] = config_module.ACSD_EVAL

for module_name in ("runtime", "baseline", "speculative", "acsd", "hf_assistant", "drift_speculative"):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

In [3]:
# Benchmark controls
DEVICE = 'cuda:0'
DRAFT_LABEL = '0.6B'
REGIME_NAME = 'deterministic'   # set to 'stochastic' if needed
K = 8
MAX_NEW_TOKENS = 64
N_PER_TASK = 4
BASE_SEED = 2026
WARMUP_NEW_TOKENS = 16

# Mode switch:
# - 'paired_on_off': only OFF vs ON (default)
# - 'single_flag_ablation': baseline OFF + one-flag-at-a-time + optional ALL_ON
BENCHMARK_MODE = 'paired_on_off'
INCLUDE_ALL_ON_IN_ABLATION = True

FLAG_NAMES = [
    'GPU_USE_SEPARATE_STREAMS',
    'GPU_PREALLOCATE_STEP_BUFFERS',
    'GPU_USE_STABLE_STEP_SHAPES',
    'GPU_TRY_CUDA_GRAPHS',
]

BASELINE_FLAGS = {name: False for name in FLAG_NAMES}
ALL_ON_FLAGS = {
    'GPU_USE_SEPARATE_STREAMS': True,
    'GPU_PREALLOCATE_STEP_BUFFERS': True,
    'GPU_USE_STABLE_STEP_SHAPES': True,
    'GPU_TRY_CUDA_GRAPHS': False,
}

def build_conditions(mode: str) -> list[dict]:
    base = {'condition': 'gpu_opt_off', **BASELINE_FLAGS}
    all_on = {'condition': 'gpu_opt_on', **ALL_ON_FLAGS}

    if mode == 'paired_on_off':
        return [base, all_on]

    if mode == 'single_flag_ablation':
        rows = [base]
        for flag in FLAG_NAMES:
            cfg = {'condition': f'only_{flag.lower()}', **BASELINE_FLAGS}
            cfg[flag] = True
            rows.append(cfg)
        if INCLUDE_ALL_ON_IN_ABLATION:
            rows.append(all_on)
        return rows

    raise ValueError(f'Unknown BENCHMARK_MODE: {mode}')

CONDITIONS = build_conditions(BENCHMARK_MODE)

print('Benchmark config:')
print(f'  device={DEVICE}, target_quant={TARGET_QUANT}, draft_quant={DRAFT_QUANT}')
print(f'  draft={DRAFT_LABEL}, regime={REGIME_NAME}, k={K}, max_new_tokens={MAX_NEW_TOKENS}')
print(f'  target={TARGET_MODEL_ID}')
print(f'  draft_model={DRAFT_MODELS[DRAFT_LABEL]}')
print(f'  n_per_task={N_PER_TASK}, warmup_new_tokens={WARMUP_NEW_TOKENS}')
print(f'  benchmark_mode={BENCHMARK_MODE}, conditions={len(CONDITIONS)}')
pd.DataFrame(CONDITIONS)[['condition', *FLAG_NAMES]]

Benchmark config:
  device=cuda:0, target_quant=fp16, draft_quant=fp16
  draft=0.6B, regime=deterministic, k=8, max_new_tokens=64
  target=Qwen/Qwen3-4B
  draft_model=Qwen/Qwen3-0.6B
  n_per_task=4, warmup_new_tokens=16
  benchmark_mode=paired_on_off, conditions=2


,condition,GPU_USE_SEPARATE_STREAMS,GPU_PREALLOCATE_STEP_BUFFERS,GPU_USE_STABLE_STEP_SHAPES,GPU_TRY_CUDA_GRAPHS
0,gpu_opt_off,False,False,False,False
1,gpu_opt_on,True,True,True,False


In [4]:
def load_prompt_rows(n_per_task: int = 4) -> list[dict]:
    rows = []
    for task in ['gsm8k', 'mmlu', 'cnndm']:
        path = manifests_dir / f'{task}_data.json'
        if not path.exists():
            continue
        with open(path, 'r') as f:
            data = json.load(f)
        for item in data[:n_per_task]:
            prompt = item.get('prompt')
            if prompt:
                rows.append({
                    'task': task,
                    'sample_id': item.get('sample_id', ''),
                    'prompt': prompt,
                })
    if not rows:
        raise RuntimeError('No prompts found in manifests/*_data.json')
    return rows

prompt_rows = load_prompt_rows(N_PER_TASK)
print(f'Loaded prompts: {len(prompt_rows)}')
pd.DataFrame(prompt_rows)[['task', 'sample_id']].head(12)

Loaded prompts: 12


,task,sample_id
0,gsm8k,gsm8k_2
1,gsm8k,gsm8k_11
2,gsm8k,gsm8k_15
3,gsm8k,gsm8k_21
4,mmlu,mmlu_abstract_algebra_0
5,mmlu,mmlu_abstract_algebra_1
6,mmlu,mmlu_abstract_algebra_2
7,mmlu,mmlu_abstract_algebra_3
8,cnndm,cnndm_3
9,cnndm,cnndm_31


In [5]:
# Load models once and reuse across both conditions.
target_model, target_tokenizer = load_model_on_device(
    TARGET_MODEL_ID,
    device=DEVICE,
    quant_mode=TARGET_QUANT,
)
draft_model, _ = load_model_on_device(
    DRAFT_MODELS[DRAFT_LABEL],
    device=DEVICE,
    quant_mode=DRAFT_QUANT,
)

print('Models loaded.')
print(f'  target: {TARGET_MODEL_ID}')
print(f'  draft : {DRAFT_MODELS[DRAFT_LABEL]}')

Loading model: Qwen/Qwen3-4B (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading model: Qwen/Qwen3-0.6B (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Models loaded.
  target: Qwen/Qwen3-4B
  draft : Qwen/Qwen3-0.6B


In [6]:
def apply_condition_flags(cfg: dict) -> None:
    spec_mod.GPU_USE_SEPARATE_STREAMS = bool(cfg['GPU_USE_SEPARATE_STREAMS'])
    spec_mod.GPU_PREALLOCATE_STEP_BUFFERS = bool(cfg['GPU_PREALLOCATE_STEP_BUFFERS'])
    spec_mod.GPU_USE_STABLE_STEP_SHAPES = bool(cfg['GPU_USE_STABLE_STEP_SHAPES'])
    spec_mod.GPU_TRY_CUDA_GRAPHS = bool(cfg['GPU_TRY_CUDA_GRAPHS'])

def run_condition(condition_cfg: dict, prompts: list[dict], base_seed: int = 0) -> pd.DataFrame:
    condition_name = condition_cfg['condition']
    apply_condition_flags(condition_cfg)
    regime = REGIMES[REGIME_NAME]

    # Warm-up (excluded from metrics).
    warm_prompt = prompts[0]['prompt']
    set_seed(base_seed)
    _ = spec_mod.speculative_decode_sample(
        target_model,
        draft_model,
        target_tokenizer,
        warm_prompt,
        max_new_tokens=min(WARMUP_NEW_TOKENS, MAX_NEW_TOKENS),
        k=K,
        temperature=regime.temperature,
        top_p=regime.top_p,
        return_timing_breakdown=True,
    )

    records = []
    for i, row in enumerate(prompts):
        seed = base_seed + i
        set_seed(seed)
        out = spec_mod.speculative_decode_sample(
            target_model,
            draft_model,
            target_tokenizer,
            row['prompt'],
            max_new_tokens=MAX_NEW_TOKENS,
            k=K,
            temperature=regime.temperature,
            top_p=regime.top_p,
            return_timing_breakdown=True,
        )

        output_text = str(out.get('output_text', ''))
        output_hash = hashlib.sha256(output_text.encode('utf-8')).hexdigest()

        records.append({
            'condition': condition_name,
            'task': row['task'],
            'sample_id': row['sample_id'],
            'seed': seed,
            'k': K,
            'max_new_tokens': MAX_NEW_TOKENS,
            'latency_s': float(out.get('latency_s', 0.0)),
            'draft_elapsed_s': float(out.get('draft_elapsed_s', 0.0)),
            'verify_elapsed_s': float(out.get('verify_elapsed_s', 0.0)),
            'ttft_ms': float(out.get('ttft_ms', 0.0)),
            'num_tokens': int(out.get('num_tokens', 0)),
            'alpha': float(out.get('alpha', 0.0)),
            'B_eff': float(out.get('B_eff', 0.0)),
            'gpu_stream_pipeline': bool(out.get('gpu_stream_pipeline', False)),
            'gpu_stable_step_shapes': bool(out.get('gpu_stable_step_shapes', False)),
            'gpu_graph_capture': out.get('gpu_graph_capture', 'unknown'),
            'output_hash': output_hash,
            'output_text': output_text,
        })

    return pd.DataFrame(records)

In [7]:
all_runs = []
for idx, cfg in enumerate(CONDITIONS):
    print(f"Running {cfg['condition']} ({idx + 1}/{len(CONDITIONS)})...")
    df_cond = run_condition(cfg, prompt_rows, base_seed=BASE_SEED)
    all_runs.append(df_cond)

df_runs = pd.concat(all_runs, ignore_index=True)
print(f'Benchmark rows: {len(df_runs)}')
print('Conditions run:', sorted(df_runs['condition'].unique()))
df_runs.head()

Running gpu_opt_off (1/2)...
Running gpu_opt_on (2/2)...
Benchmark rows: 24
Conditions run: ['gpu_opt_off', 'gpu_opt_on']


,condition,task,sample_id,seed,k,max_new_tokens,latency_s,draft_elapsed_s,verify_elapsed_s,ttft_ms,num_tokens,alpha,B_eff,gpu_stream_pipeline,gpu_stable_step_shapes,gpu_graph_capture,output_hash,output_text
0,gpu_opt_off,gsm8k,gsm8k_2,2026,8,64,3.7410,3.167077,0.545395,184.59,64,0.2529,1.95,False,False,disabled,5a07842c79ef26a93eccbdd6bac040d3ed88621b314d4d...,(Note: Profit is calculated as the selling p...
1,gpu_opt_off,gsm8k,gsm8k_11,2027,8,64,3.1906,2.696515,0.470761,183.52,64,0.3172,2.42,False,False,disabled,5bff822eee3a52337771efc9b521cadbdb760e9c2ceba1...,"\n\nOkay, let's see. Toula bought different p..."
2,gpu_opt_off,gsm8k,gsm8k_15,2028,8,64,4.5531,3.846177,0.670825,184.82,64,0.1805,1.37,False,False,disabled,0abe22d1098218e566b4f4bca0c2e8ac1f227e305868de...,(Assume that the merchant has enough money to...
3,gpu_opt_off,gsm8k,gsm8k_21,2029,8,64,2.9497,2.480636,0.444461,182.05,64,0.3433,2.56,False,False,disabled,7c385715f580f5b7c3b60124cc28ba9e89b05391a6fad7...,"Let's solve this problem step by step, and ma..."
4,gpu_opt_off,mmlu,mmlu_abstract_algebra_0,2030,8,64,2.7560,2.314503,0.419995,180.97,64,0.3840,2.82,False,False,disabled,c09c317eeff1599df4d80a5285363b405c063b4d22d6de...,"The degree of the field extension Q(sqrt(2), ..."


In [8]:
# Deterministic output-equivalence checks (hash-based).
KEYS = ['task', 'sample_id', 'seed', 'k', 'max_new_tokens']
baseline_condition = 'gpu_opt_off' if 'gpu_opt_off' in set(df_runs['condition']) else sorted(df_runs['condition'].unique())[0]

base_cols = KEYS + ['output_hash', 'output_text', 'alpha', 'B_eff']
base_df = df_runs[df_runs['condition'] == baseline_condition][base_cols].rename(columns={
    'output_hash': 'output_hash_base',
    'output_text': 'output_text_base',
    'alpha': 'alpha_base',
    'B_eff': 'B_eff_base',
})

eq_rows = []
for cond in sorted(df_runs['condition'].unique()):
    if cond == baseline_condition:
        continue
    cmp_df = df_runs[df_runs['condition'] == cond][base_cols].rename(columns={
        'output_hash': 'output_hash_cmp',
        'output_text': 'output_text_cmp',
        'alpha': 'alpha_cmp',
        'B_eff': 'B_eff_cmp',
    })
    merged = base_df.merge(cmp_df, on=KEYS, how='inner')
    if merged.empty:
        continue

    hash_match = merged['output_hash_base'] == merged['output_hash_cmp']
    alpha_match = np.isclose(merged['alpha_base'], merged['alpha_cmp'], atol=1e-8)
    beff_match = np.isclose(merged['B_eff_base'], merged['B_eff_cmp'], atol=1e-8)

    mismatch_ids = merged.loc[~hash_match, 'sample_id'].tolist()[:5]
    eq_rows.append({
        'baseline_condition': baseline_condition,
        'compare_condition': cond,
        'n_pairs': int(len(merged)),
        'n_output_mismatch': int((~hash_match).sum()),
        'output_match_rate_pct': float(hash_match.mean() * 100.0),
        'alpha_match_rate_pct': float(alpha_match.mean() * 100.0),
        'beff_match_rate_pct': float(beff_match.mean() * 100.0),
        'example_mismatch_sample_ids': ';'.join(mismatch_ids),
    })

equivalence_table = pd.DataFrame(eq_rows)
equivalence_table if not equivalence_table.empty else pd.DataFrame([{'note': 'No comparison condition available.'}])

,baseline_condition,compare_condition,n_pairs,n_output_mismatch,output_match_rate_pct,alpha_match_rate_pct,beff_match_rate_pct,example_mismatch_sample_ids
0,gpu_opt_off,gpu_opt_on,12,0,100.0,100.0,100.0,


In [9]:
# Mean delta table (paired per sample vs baseline).
METRICS = ['draft_elapsed_s', 'verify_elapsed_s', 'latency_s', 'ttft_ms']
KEYS = ['task', 'sample_id', 'seed', 'k', 'max_new_tokens']
baseline_condition = 'gpu_opt_off' if 'gpu_opt_off' in set(df_runs['condition']) else sorted(df_runs['condition'].unique())[0]

base_df = df_runs[df_runs['condition'] == baseline_condition][KEYS + METRICS].rename(columns={m: f'{m}_base' for m in METRICS})

paired_deltas = []
for cond in sorted(df_runs['condition'].unique()):
    if cond == baseline_condition:
        continue
    cmp_df = df_runs[df_runs['condition'] == cond][KEYS + METRICS].rename(columns={m: f'{m}_cmp' for m in METRICS})
    merged = base_df.merge(cmp_df, on=KEYS, how='inner')
    if merged.empty:
        continue

    for m in METRICS:
        merged[f'{m}_delta'] = merged[f'{m}_cmp'] - merged[f'{m}_base']
        merged[f'{m}_delta_pct'] = np.where(
            merged[f'{m}_base'] != 0,
            merged[f'{m}_delta'] / merged[f'{m}_base'] * 100.0,
            np.nan,
        )

    merged['baseline_condition'] = baseline_condition
    merged['compare_condition'] = cond
    paired_deltas.append(merged)

if paired_deltas:
    paired_df = pd.concat(paired_deltas, ignore_index=True)
else:
    paired_df = pd.DataFrame()

mean_rows = []
for cond in sorted(paired_df['compare_condition'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_condition'] == cond]
    for m in METRICS:
        base_mean = float(block[f'{m}_base'].mean())
        cmp_mean = float(block[f'{m}_cmp'].mean())
        abs_delta = float(block[f'{m}_delta'].mean())
        pct_delta = (abs_delta / base_mean * 100.0) if base_mean != 0 else np.nan
        mean_rows.append({
            'baseline_condition': baseline_condition,
            'compare_condition': cond,
            'metric': m,
            'baseline_mean': base_mean,
            'compare_mean': cmp_mean,
            'abs_delta_compare_minus_base': abs_delta,
            'pct_delta_compare_minus_base': pct_delta,
            'improvement_pct_lower_is_better': (-pct_delta) if pd.notna(pct_delta) else np.nan,
            'ratio_compare_over_base': (cmp_mean / base_mean) if base_mean != 0 else np.nan,
            'n_pairs': int(len(block)),
        })

mean_delta_table = pd.DataFrame(mean_rows)
mean_delta_table

,baseline_condition,compare_condition,metric,baseline_mean,compare_mean,abs_delta_compare_minus_base,pct_delta_compare_minus_base,improvement_pct_lower_is_better,ratio_compare_over_base,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,3.052591,3.038410,-0.014180,-0.464529,0.464529,0.995355,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.580288,0.587217,0.006929,1.194061,-1.194061,1.011941,12
2,gpu_opt_off,gpu_opt_on,latency_s,3.661142,3.655750,-0.005392,-0.147267,0.147267,0.998527,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,232.080000,234.128333,2.048333,0.882598,-0.882598,1.008826,12


In [10]:
# Robust delta tables: median and p90 of paired deltas.
robust_rows = []
for cond in sorted(paired_df['compare_condition'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_condition'] == cond]
    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        robust_rows.append({
            'baseline_condition': baseline_condition,
            'compare_condition': cond,
            'metric': m,
            'median_delta_compare_minus_base': float(np.median(d)),
            'p90_delta_compare_minus_base': float(np.quantile(d, 0.90)),
            'p10_delta_compare_minus_base': float(np.quantile(d, 0.10)),
            'median_abs_delta': float(np.median(np.abs(d))),
            'p90_abs_delta': float(np.quantile(np.abs(d), 0.90)),
            'n_pairs': int(len(d)),
        })

robust_delta_table = pd.DataFrame(robust_rows)

print('Median delta table:')
display(robust_delta_table[['baseline_condition','compare_condition','metric','median_delta_compare_minus_base','median_abs_delta','n_pairs']])

print('P90 delta table:')
display(robust_delta_table[['baseline_condition','compare_condition','metric','p90_delta_compare_minus_base','p90_abs_delta','n_pairs']])

Median delta table:


,baseline_condition,compare_condition,metric,median_delta_compare_minus_base,median_abs_delta,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,-0.007629,0.010786,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.007574,0.009765,12
2,gpu_opt_off,gpu_opt_on,latency_s,0.000150,0.019300,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,2.880000,4.210000,12


P90 delta table:


,baseline_condition,compare_condition,metric,p90_delta_compare_minus_base,p90_abs_delta,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,0.001236,0.054502,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.016844,0.020341,12
2,gpu_opt_off,gpu_opt_on,latency_s,0.033300,0.053840,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,4.927000,9.001000,12


In [11]:
# Paired bootstrap CI table for mean deltas.
BOOTSTRAP_SAMPLES = 20000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(BOOTSTRAP_SEED)

ci_rows = []
for cond in sorted(paired_df['compare_condition'].unique()) if not paired_df.empty else []:
    block = paired_df[paired_df['compare_condition'] == cond]
    n = len(block)
    if n == 0:
        continue

    for m in METRICS:
        d = block[f'{m}_delta'].to_numpy()
        means = np.empty(BOOTSTRAP_SAMPLES, dtype=float)
        for i in range(BOOTSTRAP_SAMPLES):
            idx = rng.integers(0, n, n)
            means[i] = float(d[idx].mean())
        lo, hi = np.quantile(means, [0.025, 0.975])
        ci_rows.append({
            'baseline_condition': baseline_condition,
            'compare_condition': cond,
            'metric': m,
            'mean_delta_compare_minus_base': float(d.mean()),
            'ci95_low': float(lo),
            'ci95_high': float(hi),
            'ci_excludes_zero': bool((lo > 0) or (hi < 0)),
            'n_pairs': int(n),
            'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
        })

bootstrap_ci_table = pd.DataFrame(ci_rows)
bootstrap_ci_table

,baseline_condition,compare_condition,metric,mean_delta_compare_minus_base,ci95_low,ci95_high,ci_excludes_zero,n_pairs,bootstrap_samples
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,-0.014180,-0.029420,0.000629,False,12,20000
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.006929,0.000446,0.012034,True,12,20000
2,gpu_opt_off,gpu_opt_on,latency_s,-0.005392,-0.022992,0.012192,False,12,20000
3,gpu_opt_off,gpu_opt_on,ttft_ms,2.048333,-0.990104,4.941687,False,12,20000


In [12]:
runs_csv = results_dir / 'gpu_opt_benchmark_runs.csv'
delta_csv = results_dir / 'gpu_opt_benchmark_delta.csv'
task_delta_csv = results_dir / 'gpu_opt_benchmark_task_delta.csv'
equiv_csv = results_dir / 'gpu_opt_benchmark_equivalence.csv'
robust_csv = results_dir / 'gpu_opt_benchmark_robust_delta.csv'
bootstrap_ci_csv = results_dir / 'gpu_opt_benchmark_bootstrap_ci.csv'

# Backward-compatible task delta summary (for OFF vs ON when present).
task_delta_rows = []
if not paired_df.empty:
    for cond in sorted(paired_df['compare_condition'].unique()):
        for task, g in paired_df[paired_df['compare_condition'] == cond].groupby('task'):
            row = {
                'baseline_condition': baseline_condition,
                'compare_condition': cond,
                'task': task,
            }
            for metric in ['draft_elapsed_s', 'verify_elapsed_s', 'latency_s']:
                base_mean = float(g[f'{metric}_base'].mean())
                cmp_mean = float(g[f'{metric}_cmp'].mean())
                row[f'{metric}_base'] = base_mean
                row[f'{metric}_compare'] = cmp_mean
                row[f'{metric}_delta_pct'] = ((cmp_mean - base_mean) / base_mean * 100.0) if base_mean != 0 else np.nan
            task_delta_rows.append(row)
task_delta_table = pd.DataFrame(task_delta_rows)

df_runs.to_csv(runs_csv, index=False)
mean_delta_table.to_csv(delta_csv, index=False)
task_delta_table.to_csv(task_delta_csv, index=False)
equivalence_table.to_csv(equiv_csv, index=False)
robust_delta_table.to_csv(robust_csv, index=False)
bootstrap_ci_table.to_csv(bootstrap_ci_csv, index=False)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
df_runs.to_csv(results_dir / f'gpu_opt_benchmark_runs_{stamp}.csv', index=False)
mean_delta_table.to_csv(results_dir / f'gpu_opt_benchmark_delta_{stamp}.csv', index=False)
bootstrap_ci_table.to_csv(results_dir / f'gpu_opt_benchmark_bootstrap_ci_{stamp}.csv', index=False)

print('Saved benchmark outputs:')
print(f'  {runs_csv}')
print(f'  {delta_csv}')
print(f'  {task_delta_csv}')
print(f'  {equiv_csv}')
print(f'  {robust_csv}')
print(f'  {bootstrap_ci_csv}')

mean_delta_table

Saved benchmark outputs:
  C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\gpu_opt_benchmark_runs.csv
  C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\gpu_opt_benchmark_delta.csv
  C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\gpu_opt_benchmark_task_delta.csv
  C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\gpu_opt_benchmark_equivalence.csv
  C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\gpu_opt_benchmark_robust_delta.csv
  C:\Working\speculative-decoding-main_v10\speculative-decoding-main\Review\results\gpu_opt_benchmark_bootstrap_ci.csv


,baseline_condition,compare_condition,metric,baseline_mean,compare_mean,abs_delta_compare_minus_base,pct_delta_compare_minus_base,improvement_pct_lower_is_better,ratio_compare_over_base,n_pairs
0,gpu_opt_off,gpu_opt_on,draft_elapsed_s,3.052591,3.038410,-0.014180,-0.464529,0.464529,0.995355,12
1,gpu_opt_off,gpu_opt_on,verify_elapsed_s,0.580288,0.587217,0.006929,1.194061,-1.194061,1.011941,12
2,gpu_opt_off,gpu_opt_on,latency_s,3.661142,3.655750,-0.005392,-0.147267,0.147267,0.998527,12
3,gpu_opt_off,gpu_opt_on,ttft_ms,232.080000,234.128333,2.048333,0.882598,-0.882598,1.008826,12
